# Phase 5: Feature Engineering — Corrected Water Cost Burden Ratio

**Purpose:** Build the `water_cost_burden_ratio = annual_water_bill / median_household_income`
using the correct FY2024-25 tiered residential tariff schedule.

Phase 3 used incorrect figures ($2.7098/kL flat, $806.08 access, $1.50/1K sewerage) sourced
from a non-residential section of the annual report. This rebuild uses the SA Water Schedule
of Fees and Charges 2024-25 residential rates.

**Inputs:**
- `data/clean/clean_master_sa2.gpkg` — Phase 3/4 master dataset with geometry
- `data/raw/2021Census_G02_SA_SA2.csv` — ABS Census 2021, median household income per SA2
- `data/raw/abs_wpi_sa_quarterly.csv` — SA WPI quarterly index (ABS 6345.0 Table 2b)

**Corrected FY2024-25 residential tariffs:**
- Tier 1: $2.251/kL for first 140 kL/year
- Tier 2: $3.214/kL for 140–520 kL/year
- Tier 3: $3.482/kL above 520 kL/year
- Fixed supply charge: $314.40/year
- Sewerage: $0.622/\$1K PV (metro) or $0.928/\$1K PV (country)
- Typical usage: 189 kL/year (SA Water stated average)

**Provider note:** Coober Pedy water is supplied by the District Council, not SA Water.
It is retained in the dataset but flagged separately.

**Dual tier outputs:**
- `burden_tier_abs`: absolute policy thresholds (Critical >4%, High 3–4%, Moderate 2–3%, Low <2%)
- `burden_tier_rel`: relative percentile rank within SA Water SA2s (top 10% = Critical, etc.)

**Outputs:**
- `data/clean/clean_master_sa2_v2.csv` — enriched master table with corrected burden ratios
- `data/clean/clean_master_sa2_v2.gpkg` — same with geometry
- `outputs/figures/fig_burden_ratio_map.html` — choropleth of burden ratio
- `outputs/figures/fig_burden_tier_abs_map.html` — absolute threshold tiers
- `outputs/figures/fig_burden_tier_rel_map.html` — relative percentile tiers
- `outputs/figures/fig_estimated_hardship_need_map.html` — estimated need per SA2

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT    = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW     = ROOT / 'data' / 'raw'
CLEAN   = ROOT / 'data' / 'clean'
FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print(f'Root: {ROOT}')

Root: C:\Users\mussa\OneDrive\Desktop\Projects\sa_water_project


## 1. Load inputs

In [2]:
master = gpd.read_file(CLEAN / 'clean_master_sa2.gpkg')
master['SA2_CODE21'] = master['SA2_CODE21'].astype(str)
print(f'Master shape: {master.shape}')
print(master[['SA2_CODE21', 'SA2_NAME21', 'SA4_NAME21']].head(3).to_string())

Master shape: (176, 19)
  SA2_CODE21      SA2_NAME21                    SA4_NAME21
0  401011001        Adelaide  Adelaide - Central and Hills
1  401011002  North Adelaide  Adelaide - Central and Hills
2  401021003  Adelaide Hills  Adelaide - Central and Hills


In [3]:
g02 = pd.read_csv(RAW / '2021Census_G02_SA_SA2.csv')
g02['SA2_CODE_2021'] = g02['SA2_CODE_2021'].astype(str)

print(f'G02 shape: {g02.shape}')
print('Nulls in income column:', g02['Median_tot_hhd_inc_weekly'].isnull().sum())
print('Income range:', g02['Median_tot_hhd_inc_weekly'].min(),
      '—', g02['Median_tot_hhd_inc_weekly'].max(), '$/week')

G02 shape: (176, 9)
Nulls in income column: 0
Income range: 0 — 3140 $/week


In [4]:
wpi = pd.read_csv(RAW / 'abs_wpi_sa_quarterly.csv')

WPI_2021_Q2 = wpi.loc[wpi['quarter'] == '2021-Q2', 'sa_wpi_index'].values[0]
WPI_2024_Q4 = wpi.loc[wpi['quarter'] == '2024-Q4', 'sa_wpi_index'].values[0]
WPI_GROWTH  = WPI_2024_Q4 / WPI_2021_Q2

print(f'WPI Q2 2021 (Census reference): {WPI_2021_Q2}')
print(f'WPI Q4 2024 (FY2024-25 midpoint): {WPI_2024_Q4}')
print(f'Growth factor: {WPI_GROWTH:.4f}  ({(WPI_GROWTH-1)*100:.1f}% total SA wage growth)')

WPI Q2 2021 (Census reference): 136.1
WPI Q4 2024 (FY2024-25 midpoint): 155.5
Growth factor: 1.1425  (14.3% total SA wage growth)


## 2. Provider type and metro/country classification

In [5]:
# Provider type: Coober Pedy water is Council-supplied (District Council of Coober Pedy).
# All other SA2s are assumed SA Water (Ceduna and other regional towns are confirmed SA Water).
def assign_provider(name):
    if name == 'Coober Pedy':
        return ('Council', 'District Council of Coober Pedy reticulates its own water supply', True)
    return ('SA Water', '', False)

provider_data = master['SA2_NAME21'].apply(assign_provider)
master['provider_type']       = provider_data.apply(lambda x: x[0])
master['provider_note']       = provider_data.apply(lambda x: x[1])
master['is_provider_confirmed'] = provider_data.apply(lambda x: x[2])

print('Provider type distribution:')
print(master['provider_type'].value_counts())
print()
print('Non-SA Water SA2s:')
print(master[master['provider_type'] != 'SA Water'][
    ['SA2_NAME21', 'SA3_NAME21', 'provider_type', 'provider_note']
].to_string(index=False))

Provider type distribution:
provider_type
SA Water    175
Council       1
Name: count, dtype: int64

Non-SA Water SA2s:
 SA2_NAME21               SA3_NAME21 provider_type                                                    provider_note
Coober Pedy Outback - North and East       Council District Council of Coober Pedy reticulates its own water supply


In [6]:
# Metro SA4s use $0.622/1K sewerage; country SA4s use $0.928/1K
METRO_SA4 = {
    'Adelaide - Central and Hills',
    'Adelaide - North',
    'Adelaide - South',
    'Adelaide - West',
}
master['is_metro'] = master['SA4_NAME21'].isin(METRO_SA4)

print('Metro / country split:')
print(master['is_metro'].value_counts().rename({True: 'Metro', False: 'Country'}))
print()
# Spot-check a few known suburbs
check = ['Adelaide', 'Elizabeth', 'Victor Harbor', 'Coober Pedy', 'Murray Bridge']
print(master[master['SA2_NAME21'].isin(check)][
    ['SA2_NAME21', 'SA4_NAME21', 'is_metro']
].to_string(index=False))

Metro / country split:
is_metro
Metro      112
Country     64
Name: count, dtype: int64

   SA2_NAME21                   SA4_NAME21  is_metro
     Adelaide Adelaide - Central and Hills      True
    Elizabeth             Adelaide - North      True
  Coober Pedy    South Australia - Outback     False
Victor Harbor South Australia - South East     False
Murray Bridge South Australia - South East     False


## 3. Corrected billing function (FY2024-25 tiered residential)

In [7]:
# FY2024-25 tiered residential tariff constants
TIER1_RATE   = 2.251   # $/kL
TIER2_RATE   = 3.214   # $/kL
TIER3_RATE   = 3.482   # $/kL
TIER1_KL     = 140.0   # kL/year — Tier 1 ceiling
TIER2_KL     = 520.0   # kL/year — Tier 2 ceiling
SUPPLY_ANNUAL = 314.40 # $/year fixed supply charge
SEWER_METRO   = 0.622  # $/year per $1,000 property value
SEWER_COUNTRY = 0.928  # $/year per $1,000 property value
TYPICAL_KL   = 189.0   # SA Water stated average residential usage
MEDIAN_PROP_K = 600.0  # $600K state-median property value baseline


def tiered_usage_charge(usage_kl, rate_multiplier=1.0):
    """Tiered usage charge for a given annual usage in kL."""
    t1 = min(usage_kl, TIER1_KL) * TIER1_RATE * rate_multiplier
    t2 = min(max(0.0, usage_kl - TIER1_KL), TIER2_KL - TIER1_KL) * TIER2_RATE * rate_multiplier
    t3 = max(0.0, usage_kl - TIER2_KL) * TIER3_RATE * rate_multiplier
    return t1 + t2 + t3


def calc_bill_scenarios(usage_kl, prop_value_k, is_metro, rate_multiplier=1.0):
    """Return all bill scenarios as a dict."""
    usage   = tiered_usage_charge(usage_kl, rate_multiplier)
    sewer   = prop_value_k * (SEWER_METRO if is_metro else SEWER_COUNTRY)
    return {
        'bill_usage_only':    round(usage, 2),
        'bill_usage_supply':  round(usage + SUPPLY_ANNUAL, 2),
        'bill_owner_metro':   round(usage + SUPPLY_ANNUAL + prop_value_k * SEWER_METRO, 2),
        'bill_owner_country': round(usage + SUPPLY_ANNUAL + prop_value_k * SEWER_COUNTRY, 2),
        'bill_owner_blended': round(usage + SUPPLY_ANNUAL + sewer, 2),
    }


# Sanity check at 189 kL / $600K property
check_metro   = calc_bill_scenarios(TYPICAL_KL, MEDIAN_PROP_K, True)
check_country = calc_bill_scenarios(TYPICAL_KL, MEDIAN_PROP_K, False)
print(f'Usage component (189 kL): ${tiered_usage_charge(TYPICAL_KL):.2f}')
print(f'  Tier 1 (140 kL × ${TIER1_RATE}): ${140*TIER1_RATE:.2f}')
print(f'  Tier 2 ( 49 kL × ${TIER2_RATE}): ${49*TIER2_RATE:.2f}')
print()
for k, v in check_metro.items():
    label = 'metro' if 'metro' in k or k == 'bill_owner_blended' else ''
    print(f'  {k}: ${v:.2f}')
print(f'  bill_owner_country (alt):  ${check_country["bill_owner_country"]:.2f}')

Usage component (189 kL): $472.63
  Tier 1 (140 kL × $2.251): $315.14
  Tier 2 ( 49 kL × $3.214): $157.49

  bill_usage_only: $472.63
  bill_usage_supply: $787.03
  bill_owner_metro: $1160.23
  bill_owner_country: $1343.83
  bill_owner_blended: $1160.23
  bill_owner_country (alt):  $1343.83


## 4. Compute WPI-adjusted median household income

In [8]:
g02['median_hhd_inc_weekly_2021'] = pd.to_numeric(g02['Median_tot_hhd_inc_weekly'], errors='coerce')
g02['median_hhd_inc_annual_2021'] = g02['median_hhd_inc_weekly_2021'] * 52
g02['median_hhd_inc_annual_adj']  = (g02['median_hhd_inc_annual_2021'] * WPI_GROWTH).round(0)
# Zero income = non-residential/industrial SA2 — treat as unknown
g02.loc[g02['median_hhd_inc_annual_adj'] <= 0, 'median_hhd_inc_annual_adj'] = float('nan')

print(f'Nulls in adjusted income: {g02["median_hhd_inc_annual_adj"].isnull().sum()}')
print('Adjusted income stats:')
print(g02['median_hhd_inc_annual_adj'].describe().round(0))

Nulls in adjusted income: 7
Adjusted income stats:
count       169.0
mean      87992.0
std       22432.0
min       43074.0
25%       72483.0
50%       84841.0
75%      103734.0
max      186554.0
Name: median_hhd_inc_annual_adj, dtype: float64


## 5. Join income to master and apply billing

In [9]:
g02_join = g02[['SA2_CODE_2021', 'median_hhd_inc_weekly_2021',
                'median_hhd_inc_annual_2021', 'median_hhd_inc_annual_adj']].copy()
g02_join = g02_join.rename(columns={'SA2_CODE_2021': 'SA2_CODE21'})

master2 = master.merge(g02_join, on='SA2_CODE21', how='left', validate='1:1')
print(f'Master shape after income join: {master2.shape}')
print(f'Income nulls: {master2["median_hhd_inc_annual_adj"].isnull().sum()}')

Master shape after income join: (176, 26)
Income nulls: 7


In [10]:
# Apply billing at 189 kL / $600K baseline; bill columns are NaN for non-SA Water SA2s
bill_rows = []
for _, row in master2.iterrows():
    if row['provider_type'] != 'SA Water':
        bill_rows.append({
            'bill_usage_only':    float('nan'),
            'bill_usage_supply':  float('nan'),
            'bill_owner_metro':   float('nan'),
            'bill_owner_country': float('nan'),
            'bill_owner_blended': float('nan'),
        })
    else:
        bill_rows.append(
            calc_bill_scenarios(TYPICAL_KL, MEDIAN_PROP_K, bool(row['is_metro']))
        )

bills_df = pd.DataFrame(bill_rows, index=master2.index)
master2 = pd.concat([master2, bills_df], axis=1)

print('Bill scenario stats (SA Water SA2s only):')
sa_water_mask = master2['provider_type'] == 'SA Water'
print(master2.loc[sa_water_mask, list(bills_df.columns)].describe().round(2))

Bill scenario stats (SA Water SA2s only):
       bill_usage_only  bill_usage_supply  bill_owner_metro  \
count           175.00             175.00            175.00   
mean            472.63             787.03           1160.23   
std               0.00               0.00              0.00   
min             472.63             787.03           1160.23   
25%             472.63             787.03           1160.23   
50%             472.63             787.03           1160.23   
75%             472.63             787.03           1160.23   
max             472.63             787.03           1160.23   

       bill_owner_country  bill_owner_blended  
count              175.00              175.00  
mean              1343.83             1226.33  
std                  0.00               88.38  
min               1343.83             1160.23  
25%               1343.83             1160.23  
50%               1343.83             1160.23  
75%               1343.83             1343.83  
max   

## 6. Compute water_cost_burden_ratio and assign tiers

In [11]:
# Primary burden ratio uses blended bill (metro or country based on SA4)
master2['water_cost_burden_ratio'] = (
    master2['bill_owner_blended'] / master2['median_hhd_inc_annual_adj']
).round(6)

# Additional scenario ratios for Power BI
master2['burden_ratio_renter']  = (master2['bill_usage_supply']  / master2['median_hhd_inc_annual_adj']).round(6)
master2['burden_ratio_owner_metro']   = (master2['bill_owner_metro']   / master2['median_hhd_inc_annual_adj']).round(6)
master2['burden_ratio_owner_country'] = (master2['bill_owner_country'] / master2['median_hhd_inc_annual_adj']).round(6)

print('Primary burden ratio (blended) stats:')
print(master2['water_cost_burden_ratio'].describe().round(4))
print()
print('As percentage of income:')
print((master2['water_cost_burden_ratio'] * 100).describe().round(2))

Primary burden ratio (blended) stats:


count    168.0000
mean       0.0149
std        0.0044
min        0.0072
25%        0.0114
50%        0.0141
75%        0.0178
max        0.0269
Name: water_cost_burden_ratio, dtype: float64

As percentage of income:
count    168.00
mean       1.49
std        0.44
min        0.72
25%        1.14
50%        1.41
75%        1.78
max        2.69
Name: water_cost_burden_ratio, dtype: float64


In [12]:
# Absolute tiers — policy thresholds; may have few Critical/High with corrected tariffs
def abs_tier(ratio):
    if pd.isna(ratio):  return 'Unknown'
    if ratio > 0.04:    return 'Critical'
    if ratio >= 0.03:   return 'High'
    if ratio >= 0.02:   return 'Moderate'
    return 'Low'

master2['burden_tier_abs'] = master2['water_cost_burden_ratio'].apply(abs_tier)

print('Absolute tier distribution (corrected tariffs):')
tier_order = ['Critical', 'High', 'Moderate', 'Low', 'Unknown']
print(master2['burden_tier_abs'].value_counts().reindex(tier_order, fill_value=0))
print()
# Show the most burdened SA2s
print('Top 15 highest burden ratio (blended bill):')
top15 = (master2[master2['water_cost_burden_ratio'].notna()]
         .nlargest(15, 'water_cost_burden_ratio')
         [['SA2_NAME21', 'SA3_NAME21', 'provider_type', 'is_metro',
           'median_hhd_inc_annual_adj', 'bill_owner_blended',
           'water_cost_burden_ratio', 'burden_tier_abs']])
top15['burden_pct'] = (top15['water_cost_burden_ratio'] * 100).round(2)
print(top15.to_string(index=False))

Absolute tier distribution (corrected tariffs):
burden_tier_abs
Critical      0
High          0
Moderate     24
Low         144
Unknown       8
Name: count, dtype: int64

Top 15 highest burden ratio (blended bill):
                     SA2_NAME21                 SA3_NAME21 provider_type  is_metro  median_hhd_inc_annual_adj  bill_owner_blended  water_cost_burden_ratio burden_tier_abs  burden_pct
                       Lonsdale                Onkaparinga      SA Water      True                    43074.0             1160.23                 0.026936        Moderate        2.69
                 Torrens Island       Port Adelaide - West      SA Water      True                    44500.0             1160.23                 0.026073        Moderate        2.61
        Yorke Peninsula - South            Yorke Peninsula      SA Water     False                    52936.0             1343.83                 0.025386        Moderate        2.54
                         Mannum          Murray and M

In [13]:
# Relative percentile tiers — always meaningful for prioritisation even when
# few SA2s cross absolute thresholds. Ranked within SA Water SA2s only.
sa_water_scored = master2[
    (master2['provider_type'] == 'SA Water') &
    master2['water_cost_burden_ratio'].notna()
].copy()

p90 = sa_water_scored['water_cost_burden_ratio'].quantile(0.90)
p75 = sa_water_scored['water_cost_burden_ratio'].quantile(0.75)
p25 = sa_water_scored['water_cost_burden_ratio'].quantile(0.25)

def rel_tier(ratio):
    if pd.isna(ratio):    return 'Unknown'
    if ratio >= p90:      return 'Critical'   # top 10%
    if ratio >= p75:      return 'High'        # 75th–90th
    if ratio >= p25:      return 'Moderate'    # 25th–75th
    return 'Low'                               # bottom 25%

master2['burden_tier_rel'] = master2.apply(
    lambda r: rel_tier(r['water_cost_burden_ratio'])
    if r['provider_type'] == 'SA Water' else 'Unknown',
    axis=1
)

print(f'Percentile cutoffs: p25={p25:.4f}  p75={p75:.4f}  p90={p90:.4f}')
print()
print('Relative tier distribution:')
print(master2['burden_tier_rel'].value_counts().reindex(tier_order, fill_value=0))
print()
print('Relative Critical SA2s (top 10% most burdened):')
crit_rel = master2[master2['burden_tier_rel'] == 'Critical'][[
    'SA2_NAME21', 'SA3_NAME21', 'is_metro',
    'bill_owner_blended', 'median_hhd_inc_annual_adj', 'water_cost_burden_ratio'
]].sort_values('water_cost_burden_ratio', ascending=False)
crit_rel['burden_pct'] = (crit_rel['water_cost_burden_ratio'] * 100).round(2)
print(crit_rel.to_string(index=False))

Percentile cutoffs: p25=0.0114  p75=0.0178  p90=0.0216



Relative tier distribution:
burden_tier_rel
Critical    17
High        25
Moderate    84
Low         42
Unknown      8
Name: count, dtype: int64

Relative Critical SA2s (top 10% most burdened):
                     SA2_NAME21                 SA3_NAME21  is_metro  bill_owner_blended  median_hhd_inc_annual_adj  water_cost_burden_ratio  burden_pct
                       Lonsdale                Onkaparinga      True             1160.23                    43074.0                 0.026936        2.69
                 Torrens Island       Port Adelaide - West      True             1160.23                    44500.0                 0.026073        2.61
        Yorke Peninsula - South            Yorke Peninsula     False             1343.83                    52936.0                 0.025386        2.54
                         Mannum          Murray and Mallee     False             1343.83                    53530.0                 0.025104        2.51
                       Wallaroo        

In [14]:
# Summary statistics by relative tier
valid = master2[master2['water_cost_burden_ratio'].notna() &
                (master2['provider_type'] == 'SA Water')].copy()

summary = (
    valid.groupby('burden_tier_rel')
    .agg(
        count=('SA2_CODE21', 'count'),
        population=('population', 'sum'),
        income_median=('median_hhd_inc_annual_adj', 'median'),
        bill_median=('bill_owner_blended', 'median'),
        ratio_min=('water_cost_burden_ratio', 'min'),
        ratio_median=('water_cost_burden_ratio', 'median'),
        ratio_max=('water_cost_burden_ratio', 'max'),
    )
    .reindex([t for t in ['Critical','High','Moderate','Low'] if t in valid['burden_tier_rel'].unique()])
    .round(4)
)
summary['burden_pct_median'] = (summary['ratio_median'] * 100).round(2)
print('Relative tier summary:')
print(summary.to_string())

Relative tier summary:


                 count  population  income_median  bill_median  ratio_min  ratio_median  ratio_max  burden_pct_median
burden_tier_rel                                                                                                      
Critical            17    111107.0        57333.0      1343.83     0.0217        0.0234     0.0269               2.34
High                25    182198.0        68205.0      1343.83     0.0179        0.0192     0.0216               1.92
Moderate            84   1051699.0        85227.0      1160.23     0.0115        0.0141     0.0177               1.41
Low                 42    431534.0       114517.0      1160.23     0.0072        0.0101     0.0112               1.01


## 7. Estimated hardship need

SA Water's payment assistance program (FY2023-24): 2,732 customers, $2,590 avg debt.
No suburb-level enrollment data is publicly available. This section computes an
**estimated need score** — how many assistance customers each SA2 *would expect* if
uptake were proportional to tier and population. This is a **prioritisation proxy**,
not a measurement of actual program under-service.

In [15]:
TOTAL_HARDSHIP_CUSTOMERS = 2732  # FY2023-24 Annual Report

tier_weights = {'Critical': 4.0, 'High': 2.0, 'Moderate': 1.0, 'Low': 0.25, 'Unknown': 0.0}

# Use relative tier for weighting (it is always populated for SA Water SA2s)
valid2 = master2[
    (master2['provider_type'] == 'SA Water') &
    master2['water_cost_burden_ratio'].notna()
].copy()
valid2['tier_weight'] = valid2['burden_tier_rel'].map(tier_weights)
valid2['hardship_score_raw'] = valid2['population'].fillna(0) * valid2['tier_weight']

total_weighted = valid2['hardship_score_raw'].sum()
valid2['estimated_hardship_need'] = (
    valid2['hardship_score_raw'] / total_weighted * TOTAL_HARDSHIP_CUSTOMERS
).round(1)

valid2['burden_ratio_norm'] = valid2.groupby('burden_tier_rel')['water_cost_burden_ratio'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)
valid2['hardship_priority_score'] = (
    valid2['tier_weight'] * (1 + valid2['burden_ratio_norm'])
).round(4)

print('Estimated hardship need by relative tier:')
print(valid2.groupby('burden_tier_rel')['estimated_hardship_need']
          .sum().reindex(['Critical','High','Moderate','Low']).round(1))
print(f'\nTotal: {valid2["estimated_hardship_need"].sum():.0f}  (should ≈ {TOTAL_HARDSHIP_CUSTOMERS})')
print()
print('Top 10 SA2s by estimated hardship need:')
top10 = valid2.nlargest(10, 'estimated_hardship_need')[[
    'SA2_NAME21', 'SA3_NAME21', 'burden_tier_rel', 'burden_tier_abs',
    'water_cost_burden_ratio', 'population', 'estimated_hardship_need'
]]
top10['burden_pct'] = (top10['water_cost_burden_ratio'] * 100).round(2)
print(top10.to_string(index=False))

Estimated hardship need by relative tier:
burden_tier_rel
Critical     616.8
High         505.6
Moderate    1459.6
Low          149.5
Name: estimated_hardship_need, dtype: float64

Total: 2732  (should ≈ 2732)

Top 10 SA2s by estimated hardship need:
             SA2_NAME21                    SA3_NAME21 burden_tier_rel burden_tier_abs  water_cost_burden_ratio  population  estimated_hardship_need  burden_pct
          Victor Harbor    Fleurieu - Kangaroo Island        Critical        Moderate                 0.023151     15847.0                     88.0        2.32
             Port Pirie                     Mid North        Critical        Moderate                 0.021666     13896.0                     77.1        2.17
   Goolwa - Port Elliot    Fleurieu - Kangaroo Island        Critical        Moderate                 0.023318     12516.0                     69.5        2.33
              Elizabeth                      Playford        Critical        Moderate                 0.02361

## 8. Merge enriched columns and build final dataset

In [16]:
# Merge hardship columns back to full master
hardship_cols = ['SA2_CODE21', 'estimated_hardship_need', 'hardship_priority_score']
master_final = master2.merge(
    valid2[hardship_cols], on='SA2_CODE21', how='left'
)

# Reproject to WGS84 for Plotly
gdf_wgs = master_final.to_crs(epsg=4326)
geojson = json.loads(gdf_wgs.to_json())

print(f'Final GDF shape: {gdf_wgs.shape}')
print(f'Null counts for key columns:')
key_cols = ['water_cost_burden_ratio', 'burden_tier_abs', 'burden_tier_rel',
            'estimated_hardship_need', 'median_hhd_inc_annual_adj']
print(gdf_wgs[key_cols].isnull().sum())

Final GDF shape: (176, 39)
Null counts for key columns:
water_cost_burden_ratio      8
burden_tier_abs              0
burden_tier_rel              0
estimated_hardship_need      8
median_hhd_inc_annual_adj    7
dtype: int64


## 9. Choropleth maps

In [17]:
tier_colours = {
    'Critical': '#d32f2f', 'High': '#f57c00',
    'Moderate': '#fbc02d', 'Low': '#388e3c', 'Unknown': '#bdbdbd'
}
gdf_sa_water = gdf_wgs[
    (gdf_wgs['provider_type'] == 'SA Water') &
    gdf_wgs['water_cost_burden_ratio'].notna()
].copy()

# Map 1: Burden ratio (continuous)
fig1 = px.choropleth_mapbox(
    gdf_sa_water,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='water_cost_burden_ratio',
    color_continuous_scale='RdYlGn_r',
    range_color=[
        gdf_sa_water['water_cost_burden_ratio'].quantile(0.05),
        gdf_sa_water['water_cost_burden_ratio'].quantile(0.95),
    ],
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'burden_tier_abs': True,
        'burden_tier_rel': True,
        'water_cost_burden_ratio': ':.4f',
        'median_hhd_inc_annual_adj': ':,.0f',
        'bill_owner_blended': ':,.2f',
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Water Cost Burden Ratio — SA2 Areas, South Australia (FY2024-25, corrected tariff)',
    labels={'water_cost_burden_ratio': 'Burden Ratio'},
)
fig1.update_layout(margin={'r':0,'t':40,'l':0,'b':0}, height=700)
fig1.write_html(FIGURES / 'fig_burden_ratio_map.html')
print('Saved fig_burden_ratio_map.html')

C:\Users\mussa\AppData\Local\Temp\ipykernel_52080\1116215833.py:11: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig1 = px.choropleth_mapbox(


Saved fig_burden_ratio_map.html


In [18]:
# Map 2: Absolute tier
fig2 = px.choropleth_mapbox(
    gdf_wgs,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='burden_tier_abs',
    color_discrete_map=tier_colours,
    category_orders={'burden_tier_abs': tier_order},
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'provider_type': True,
        'water_cost_burden_ratio': ':.4f',
        'median_hhd_inc_annual_adj': ':,.0f',
        'burden_tier_abs': False,
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Water Stress Tiers (Absolute Thresholds, corrected FY2024-25 tariff) — South Australia',
)
fig2.update_layout(margin={'r':0,'t':40,'l':0,'b':0}, height=700)
fig2.write_html(FIGURES / 'fig_burden_tier_abs_map.html')
print('Saved fig_burden_tier_abs_map.html')

C:\Users\mussa\AppData\Local\Temp\ipykernel_52080\850968061.py:2: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = px.choropleth_mapbox(


Saved fig_burden_tier_abs_map.html


In [19]:
# Map 3: Relative percentile tier
fig3 = px.choropleth_mapbox(
    gdf_wgs,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='burden_tier_rel',
    color_discrete_map=tier_colours,
    category_orders={'burden_tier_rel': tier_order},
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'provider_type': True,
        'water_cost_burden_ratio': ':.4f',
        'median_hhd_inc_annual_adj': ':,.0f',
        'burden_tier_rel': False,
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Water Stress Tiers (Relative Percentile Rank) — South Australia (FY2024-25)',
)
fig3.update_layout(margin={'r':0,'t':40,'l':0,'b':0}, height=700)
fig3.write_html(FIGURES / 'fig_burden_tier_rel_map.html')
print('Saved fig_burden_tier_rel_map.html')

C:\Users\mussa\AppData\Local\Temp\ipykernel_52080\939117838.py:2: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig3 = px.choropleth_mapbox(


Saved fig_burden_tier_rel_map.html


In [20]:
# Map 4: Estimated hardship need
gdf_need = gdf_wgs[gdf_wgs['estimated_hardship_need'].notna()].copy()
fig4 = px.choropleth_mapbox(
    gdf_need,
    geojson=geojson,
    locations='SA2_CODE21',
    featureidkey='properties.SA2_CODE21',
    color='estimated_hardship_need',
    color_continuous_scale='Reds',
    hover_name='SA2_NAME21',
    hover_data={
        'SA3_NAME21': True,
        'burden_tier_rel': True,
        'estimated_hardship_need': ':.0f',
        'water_cost_burden_ratio': ':.4f',
        'population': ':,.0f',
        'SA2_CODE21': False,
    },
    mapbox_style='carto-positron',
    center={'lat': -30.5, 'lon': 135.5},
    zoom=4.5,
    opacity=0.75,
    title='Estimated Hardship Assistance Need — SA2 Areas, South Australia (proxy estimate)',
    labels={'estimated_hardship_need': 'Est. Customers'},
)
fig4.update_layout(margin={'r':0,'t':40,'l':0,'b':0}, height=700)
fig4.write_html(FIGURES / 'fig_estimated_hardship_need_map.html')
print('Saved fig_estimated_hardship_need_map.html')

C:\Users\mussa\AppData\Local\Temp\ipykernel_52080\710987040.py:3: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig4 = px.choropleth_mapbox(


Saved fig_estimated_hardship_need_map.html


## 10. Save final dataset

In [21]:
FINAL_COLS = [
    'SA2_CODE21', 'SA2_NAME21', 'SA3_NAME21', 'SA4_NAME21',
    'population', 'AREASQKM21',
    # Provider
    'provider_type', 'provider_note', 'is_provider_confirmed',
    # Location
    'is_metro',
    # SEIFA indexes
    'irsd_score', 'irsd_decile',
    'irsad_score', 'irsad_decile',
    'ier_score', 'ier_decile',
    'ieo_score', 'ieo_decile',
    # Income
    'median_hhd_inc_weekly_2021',
    'median_hhd_inc_annual_2021',
    'median_hhd_inc_annual_adj',
    # Bill scenarios (corrected FY2024-25 tiered tariff)
    'bill_usage_only',
    'bill_usage_supply',
    'bill_owner_metro',
    'bill_owner_country',
    'bill_owner_blended',
    # Burden ratios
    'water_cost_burden_ratio',
    'burden_ratio_renter',
    'burden_ratio_owner_metro',
    'burden_ratio_owner_country',
    # Dual stress tiers
    'burden_tier_abs',
    'burden_tier_rel',
    # Hardship (renamed from gap — this is estimated need, not measured gap)
    'estimated_hardship_need',
    'hardship_priority_score',
    # Legacy IER-proxy metrics (kept as ML features and reference)
    'water_stress_index',
    'stress_pct_rank',
    'stress_tier',
    # Geometry
    'geometry',
]

master_out = master_final[FINAL_COLS].copy()
print(f'Final shape: {master_out.shape}')
print(master_out.dtypes)

Final shape: (176, 38)
SA2_CODE21                      object
SA2_NAME21                      object
SA3_NAME21                      object
SA4_NAME21                      object
population                     float64
AREASQKM21                     float64
provider_type                   object
provider_note                   object
is_provider_confirmed             bool
is_metro                          bool
irsd_score                     float64
irsd_decile                    float64
irsad_score                    float64
irsad_decile                   float64
ier_score                      float64
ier_decile                     float64
ieo_score                      float64
ieo_decile                     float64
median_hhd_inc_weekly_2021       int64
median_hhd_inc_annual_2021       int64
median_hhd_inc_annual_adj      float64
bill_usage_only                float64
bill_usage_supply              float64
bill_owner_metro               float64
bill_owner_country             float64
bi

In [22]:
out_csv  = CLEAN / 'clean_master_sa2_v2.csv'
out_gpkg = CLEAN / 'clean_master_sa2_v2.gpkg'

master_out.drop(columns=['geometry']).to_csv(out_csv, index=False)
print(f'Saved: {out_csv.name}  ({out_csv.stat().st_size // 1024} KB)')

gpd.GeoDataFrame(master_out, geometry='geometry', crs=master_final.crs).to_file(out_gpkg, driver='GPKG')
print(f'Saved: {out_gpkg.name}  ({out_gpkg.stat().st_size // 1024} KB)')

Saved: clean_master_sa2_v2.csv  (48 KB)


Saved: clean_master_sa2_v2.gpkg  (3152 KB)


In [23]:
print('=== PHASE 5 COMPLETE (corrected billing engine) ===')
print(f'Tariff: FY2024-25 tiered residential')
print(f'  T1: ${TIER1_RATE}/kL ≤{TIER1_KL}kL  |  T2: ${TIER2_RATE}/kL {TIER1_KL}–{TIER2_KL}kL  |  T3: ${TIER3_RATE}/kL >{TIER2_KL}kL')
print(f'  Supply: ${SUPPLY_ANNUAL}/yr  |  Sewer metro: ${SEWER_METRO}/$1K  |  Sewer country: ${SEWER_COUNTRY}/$1K')
print(f'  Typical usage: {TYPICAL_KL} kL/yr (SA Water average)  |  Baseline property: ${MEDIAN_PROP_K}K')
print()
print(f'Corrected typical bills:')
print(f'  Owner-metro:   ${check_metro["bill_owner_blended"]:.2f}/yr')
print(f'  Owner-country: ${check_country["bill_owner_blended"]:.2f}/yr')
print(f'  Renter:        ${check_metro["bill_usage_supply"]:.2f}/yr')
print()
print('Burden tier distributions:')
print('  Absolute (policy thresholds):')
print('  ' + str(master_out['burden_tier_abs'].value_counts().reindex(tier_order, fill_value=0).to_dict()))
print('  Relative (percentile ranks — SA Water SA2s only):')
print('  ' + str(master_out['burden_tier_rel'].value_counts().reindex(tier_order, fill_value=0).to_dict()))
print()
print('Non-SA Water SA2s:')
print(master_out[master_out['provider_type'] != 'SA Water'][['SA2_NAME21','provider_type']].to_string(index=False))
print()
print('Next: Phase 6 — Genuine proxy-risk ML model (SEIFA features only, no income/bill leakage)')

=== PHASE 5 COMPLETE (corrected billing engine) ===
Tariff: FY2024-25 tiered residential
  T1: $2.251/kL ≤140.0kL  |  T2: $3.214/kL 140.0–520.0kL  |  T3: $3.482/kL >520.0kL
  Supply: $314.4/yr  |  Sewer metro: $0.622/$1K  |  Sewer country: $0.928/$1K
  Typical usage: 189.0 kL/yr (SA Water average)  |  Baseline property: $600.0K

Corrected typical bills:
  Owner-metro:   $1160.23/yr
  Owner-country: $1343.83/yr
  Renter:        $787.03/yr

Burden tier distributions:
  Absolute (policy thresholds):
  {'Critical': 0, 'High': 0, 'Moderate': 24, 'Low': 144, 'Unknown': 8}
  Relative (percentile ranks — SA Water SA2s only):
  {'Critical': 17, 'High': 25, 'Moderate': 84, 'Low': 42, 'Unknown': 8}

Non-SA Water SA2s:
 SA2_NAME21 provider_type
Coober Pedy       Council

Next: Phase 6 — Genuine proxy-risk ML model (SEIFA features only, no income/bill leakage)
